# Polymarket: finns det systematiskt felprissatta kontrakt?

**Frågan:** när ett kontrakt kostade X öre, vann det verkligen X % av gångerna?

Om folk systematiskt överskattar vissa utfall (optimism, lotteritänkande, favoriter) syns det som en skillnad mellan pris och faktisk vinstfrekvens. En sådan skillnad är bara värd något om den

1. är **större än avgifter och spread**,
2. är **statistiskt säker** (inte slump), och
3. **håller över tid** (finns både i äldre och nyare data).

Notebooken läser bara offentlig data från Polymarket. Inga konton, nycklar eller order.

## Setup

Lokalt hittas `pmcal` i repot. I **Google Colab** hämtas bara notebooken, så cellen nedan klonar repot först. Repot är privat, så Colab behöver en GitHub-token:

1. Skapa en token på GitHub (Settings → Developer settings → Fine-grained tokens) med läsrätt (*Contents: Read*) för `HEVI-SE/PM`.
2. I Colab: klicka på nyckelikonen 🔑 i vänstermenyn, lägg till en secret med namnet `GITHUB_TOKEN` och slå på *Notebook access*.

In [ ]:
import sys, os, subprocess

REPO = "HEVI-SE/PM"
BRANCH = "claude/hopeful-mccarthy-bkmowc"

if "google.colab" in sys.modules and not os.path.exists("/content/PM"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        from getpass import getpass  # ingen secret: fråga istället (visas inte och sparas inte)
        token = getpass("GitHub-token med läsrätt till HEVI-SE/PM: ")
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", "/content/PM"], check=True)
if "google.colab" in sys.modules:
    os.chdir("/content/PM/notebooks")

sys.path.insert(0, os.path.abspath(".."))  # så att pmcal hittas från notebooks/

import pandas as pd
import matplotlib.pyplot as plt
import pmcal

pd.set_option("display.float_format", "{:.3f}".format)

## Inställningar

Varje källa är en grupp marknader att jämföra. `contains` filtrerar på text i fråga/slug, `tag` hämtar en kategori via Polymarkets events.

- **horizons:** hur långt före stängning priset läses. För 5-minutersmarknader är minuter rimligt, för politik timmar eller dagar.
- **fee:** avgift som andel av insatsen. Snabba kryptomarknader har taker-avgift (upp till ca 1,5 % nära 50 öre), de flesta andra har ingen. Kontrollera aktuella avgifter.
- **SLIPPAGE:** påslag på priset för att du i praktiken köper på säljsidan av orderboken.

Kontrollera vad BTC-marknaderna faktiskt heter på Polymarket och justera `contains` vid behov.

In [ ]:
SOURCES = [
    {"name": "BTC upp/ner", "contains": "bitcoin up or down", "horizons": ["1m", "3m"], "fee": 0.0156},
    {"name": "Politik",     "tag": "politics",                "horizons": ["1h", "24h", "7d"], "fee": 0.0},
    {"name": "Sport",       "tag": "sports",                  "horizons": ["1h", "24h"], "fee": 0.0},
]
N_MARKETS = 500      # per källa; börja lågt (t.ex. 50) för att testa
MIN_VOLUME = 1000    # USD, filtrerar bort marknader nästan utan handel
SLIPPAGE = 0.01

## Hämta data

Svaren cachas i `.pm_cache/`, så en omkörning går snabbt. Första körningen kan ta en stund: en förfrågan per kontrakt och horisont.

In [ ]:
frames = []
for s in SOURCES:
    print(f"== {s['name']} ==")
    if s.get("tag"):
        markets = pmcal.fetch_markets_by_tag(s["tag"], n=N_MARKETS, min_volume=MIN_VOLUME)
    else:
        markets = pmcal.fetch_markets(n=N_MARKETS, contains=s.get("contains"), min_volume=MIN_VOLUME)
    for h in s["horizons"]:
        rows = pmcal.contracts(markets, h)
        frames.append(pd.DataFrame(rows).assign(source=s["name"], fee=s["fee"]))

df = pd.concat(frames, ignore_index=True)
df["group"] = df["source"] + " / " + df["horizon"]
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/contracts.csv", index=False)
df.groupby("group").agg(kontrakt=("price", "size"), marknader=("market_id", "nunique"))

## Kalibrering

En rad per prisintervall och grupp. Läs så här:

- **avvikelse** = vinstfrekvens − snittpris. Positivt betyder att kontrakten var för billiga.
- **ki_lag / ki_hog** = 95 % konfidensintervall för vinstfrekvensen.
- **ev_per_krona** = avkastning per insatt krona efter avgift och spread.
- **edge** = True bara om EV är positivt *och* hela konfidensintervallet ligger över priset plus spread.

Varning: i de billigaste intervallen (0–5 öre) är brus enormt. Några enstaka vinster ger högt EV av ren slump. Lita bara på `edge`.

In [ ]:
def calibrate_all(d):
    parts = []
    for g, sub in d.groupby("group"):
        t = pmcal.calibrate(sub, fee=sub["fee"].iloc[0], slippage=SLIPPAGE)
        parts.append(t.assign(group=g))
    return pd.concat(parts, ignore_index=True)

cal = calibrate_all(df)
cal

In [ ]:
summary = df.groupby("group").apply(pmcal.brier, include_groups=False).rename("brier").to_frame()
summary["kontrakt"] = df.groupby("group").size()
summary["intervall_med_edge"] = cal.groupby("group")["edge"].sum()
summary  # lägre Brier = bättre prissatt marknad

## Kalibreringskurvor

Punkter på diagonalen betyder rätt prissatt. Över diagonalen: för billigt. Under: för dyrt.

In [ ]:
groups = cal["group"].unique()
cols = 3
fig, axes = plt.subplots(-(-len(groups) // cols), cols, figsize=(5 * cols, 4.5 * -(-len(groups) // cols)), squeeze=False)
for ax, g in zip(axes.flat, groups):
    t = cal[cal["group"] == g]
    ax.plot([0, 1], [0, 1], color="grey", lw=1, ls="--")
    ax.errorbar(t["snittpris"], t["vinstfrekvens"],
                yerr=[t["vinstfrekvens"] - t["ki_lag"], t["ki_hog"] - t["vinstfrekvens"]],
                fmt="o", capsize=3)
    for _, r in t[t["edge"]].iterrows():
        ax.annotate("edge", (r["snittpris"], r["vinstfrekvens"]), textcoords="offset points", xytext=(5, -10))
    ax.set(title=g, xlabel="pris", ylabel="vinstfrekvens", xlim=(0, 1), ylim=(0, 1))
for ax in axes.flat[len(groups):]:
    ax.axis("off")
plt.tight_layout()

## Avkastning per prisintervall efter avgifter

In [ ]:
pivot = cal.pivot(index="intervall", columns="group", values="ev_per_krona")
ax = pivot.plot.bar(figsize=(12, 5))
ax.axhline(0, color="black", lw=1)
ax.set(ylabel="EV per insatt krona", title="Förväntad avkastning efter avgift och spread")
plt.tight_layout()

## Håller det över tid?

En verklig felprissättning ska finnas både i äldre och nyare data. Vi delar varje grupp på mitten efter stängningsdatum och kalibrerar halvorna var för sig. Bara intervall med `edge` i **båda** halvorna är intressanta.

In [ ]:
def halves(d):
    parts = []
    for g, sub in d.groupby("group"):
        cut = sub["end"].median()
        for name, half in [("äldre", sub[sub["end"] <= cut]), ("nyare", sub[sub["end"] > cut])]:
            t = pmcal.calibrate(half, fee=sub["fee"].iloc[0], slippage=SLIPPAGE)
            parts.append(t.assign(group=g, halva=name))
    return pd.concat(parts, ignore_index=True)

h = halves(df)
stable = (h.pivot_table(index=["group", "intervall"], columns="halva", values="edge", aggfunc="first")
            .fillna(False).astype(bool))
stable["båda"] = stable.get("äldre", False) & stable.get("nyare", False)
stable[stable["båda"]] if stable["båda"].any() else "Ingen stabil edge i någon grupp."

## Tolkning

- **Ingen `edge` någonstans:** marknaderna är effektiva nog för en långsam aktör. Det är ett användbart svar, och billigt att få.
- **`edge` i bara ena halvan:** troligen slump eller en tillfällig period. Bygg inte på det.
- **Stabil `edge`:** värt att gå vidare. Kontrollera då orderbokens djup (hur mycket går att köpa till det priset), hur länge kapitalet låses, och kör strategin på papper innan riktiga pengar.

Kom ihåg att flera intervall och grupper testas samtidigt. Med många tester dyker enstaka falska träffar upp av ren slump.